In [1]:
import wrds
import pandas as pd

In [3]:
db = wrds.Connection()

WRDS recommends setting up a .pgpass file.
Created .pgpass file successfully.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done


In [3]:
# db.list_libraries()
# db.list_tables(library='secsamp')
# db.list_tables(library='secsamp_all')
# db.list_tables(library='wrdssec')


In [4]:
# query = """
# WITH filings AS (
#     SELECT
#         cik,
#         form,
#         fdate,
#         accession
#     FROM wrdssec.wrds_forms
#     WHERE form = '10-K'
#     AND fdate >= '2000-01-01'
# ),

# link AS (
#     SELECT
#         cik,
#         gvkey
#     FROM wrdssec.wciklink_gvkey
# ),

# ccm AS (
#     SELECT
#         gvkey,
#         lpermno AS permno
#     FROM crsp.ccmxpf_linktable
#     WHERE linktype IN ('LU','LC')
#     AND linkprim IN ('P','C')
# ),

# fund AS (
#     SELECT
#         gvkey,
#         datadate,
#         at,
#         sale,
#         ni
#     FROM comp.funda
# ),

# returns AS (
#     SELECT
#         permno,
#         date,
#         ret
#     FROM crsp.msf
# )

# SELECT
#     f.cik,
#     f.form,
#     f.fdate,
#     f.accession,

#     l.gvkey,
#     c.permno,

#     r.ret AS next_month_ret,

#     comp.datadate,
#     comp.at,
#     comp.sale,
#     comp.ni

# FROM filings f

# LEFT JOIN link l
#     ON f.cik = l.cik

# LEFT JOIN ccm c
#     ON l.gvkey = c.gvkey

# LEFT JOIN returns r
#     ON c.permno = r.permno
#     AND r.date = DATE_TRUNC('month', f.fdate) + INTERVAL '1 month'

# LEFT JOIN LATERAL (
#     SELECT
#         datadate,
#         at,
#         sale,
#         ni
#     FROM fund
#     WHERE fund.gvkey = l.gvkey
#     AND datadate <= f.fdate
#     ORDER BY datadate DESC
#     LIMIT 1
# ) comp ON TRUE
# """

# sec_df = db.raw_sql(query)

In [5]:
# sec_df.head()

In [6]:
# query_text = """
# SELECT cik, accession, sentiment, uncertainty
# FROM wrdssec.wrds_nlpsa
# """

# text = db.raw_sql(query_text)

# df = df.merge(text, on=["cik","accession"], how="left")

In [ ]:
query = """
WITH filings AS (
    SELECT
        cik,
        accession,
        fdate
    FROM wrdssec.wrds_forms
    WHERE form='10-K'
    AND fdate >= '2000-01-01'
),

cik_link AS (
    SELECT DISTINCT
        cik,
        gvkey
    FROM wrdssec.wciklink_gvkey
),

ccm AS (
    SELECT
        gvkey,
        lpermno AS permno,
        linkdt,
        linkenddt
    FROM crsp.ccmxpf_linktable
    WHERE linktype IN ('LU','LC')
    AND linkprim IN ('P','C')
),

returns AS (
    SELECT
        permno,
        date,
        ret
    FROM crsp.msf
),

fund AS (
    SELECT
        gvkey,
        datadate,
        at,
        sale,
        ni
    FROM comp.funda
)

SELECT
    f.cik,
    f.accession,
    f.fdate,
    ck.gvkey,
    c.permno,

    r.date AS return_month,
    r.ret AS monthly_return,

    comp.datadate,
    comp.at,
    comp.sale,
    comp.ni

FROM filings f

LEFT JOIN cik_link ck
    ON f.cik = ck.cik

LEFT JOIN ccm c
    ON ck.gvkey = c.gvkey
    AND f.fdate BETWEEN c.linkdt AND COALESCE(c.linkenddt, '2100-01-01')

LEFT JOIN returns r
    ON c.permno = r.permno
    AND r.date = DATE_TRUNC('month', f.fdate) + INTERVAL '1 month' - INTERVAL '1 day'

LEFT JOIN LATERAL (
    SELECT datadate, at, sale, ni
    FROM fund
    WHERE fund.gvkey = ck.gvkey
    AND datadate <= f.fdate
    ORDER BY datadate DESC
    LIMIT 1
) comp ON TRUE
"""

df = db.raw_sql(query)

df.to_csv("lazy_prices_dataset.csv")

In [ ]:
import numpy as np


df = pd.read_csv("lazy_prices_dataset.csv")
accessions = df.accession.unique()

batch_size = 5000

for year in range(2010, 2025):

    print("Year:", year)

    for i in range(0, len(accessions), batch_size):

        batch = tuple(accessions[i:i+batch_size])

        query = f"""
        SELECT accession, word, noccur
        FROM wrdssec_bow.bow_{year}
        WHERE accession IN {batch}
        """

        chunk = pd.read_sql_query(query, db.connection)

        chunk.to_parquet(
            f"bow_{year}_batch_{i}.parquet",
            compression="snappy"
        )

        print(f"Saved batch {i}")

In [12]:
meta = pd.read_csv("lazy_prices_dataset.csv")[["accession","cik","fdate"]]

meta = meta.sort_values(["cik","fdate"])

meta["prev_accession"] = meta.groupby("cik")["accession"].shift(1)

pairs = meta.dropna()

In [13]:
import polars as pl

bow = pl.scan_parquet("bow_*_batch_*.parquet")

In [15]:
norms = (
    bow
    .group_by("accession")
    .agg((pl.col("noccur")**2).sum().sqrt().alias("norm"))
)

In [17]:

pairs_pl = pl.from_pandas(pairs).lazy()

sim = (
    pairs_pl

    # join words from current filing
    .join(
        bow,
        on="accession",
        how="left"
    )
    .rename({
        "word": "word_a",
        "noccur": "count_a"
    })

    # join words from previous filing
    .join(
        bow,
        left_on=["prev_accession", "word_a"],
        right_on=["accession", "word"],
        how="left"
    )
    .rename({
        "noccur": "count_b"
    })

    # dot product
    .with_columns(
        (pl.col("count_a") * pl.col("count_b")).alias("dot")
    )

    # sum dot products
    .group_by(["accession", "prev_accession"])
    .agg(
        pl.col("dot").sum().alias("dot_product")
    )
)

In [18]:
norms = (
    bow
    .group_by("accession")
    .agg(
        (pl.col("noccur")**2).sum().sqrt().alias("norm")
    )
)

In [19]:
sim = (
    sim
    .join(norms, on="accession")
    .rename({"norm": "norm_a"})
    .join(norms, left_on="prev_accession", right_on="accession")
    .rename({"norm": "norm_b"})
    .with_columns(
        (pl.col("dot_product") /
        (pl.col("norm_a") * pl.col("norm_b")))
        .alias("cosine_similarity")
    )
)

In [20]:
sim_df = sim.collect().to_pandas()

: 

In [13]:
query = """
SELECT DISTINCT c.cik, c.conm, cr.permno,
       ABS(cr.prc) * cr.shrout AS market_cap,
       cr.prc,
       cr.vol * ABS(cr.prc) AS dollar_volume
FROM comp.company c
JOIN crsp.ccmxpf_linktable l
  ON c.gvkey = l.gvkey
JOIN crsp.dsf cr
  ON l.lpermno = cr.permno
WHERE c.cik IS NOT NULL
  AND c.fic = 'USA'
  AND cr.date = '2023-12-29'   -- pick a snapshot date
  AND l.linktype IN ('LU', 'LC')
  AND l.linkprim IN ('P', 'C')
  AND ABS(cr.prc) >= 5
  AND (ABS(cr.prc) * cr.shrout) > 2e6
  AND (cr.vol * ABS(cr.prc)) > 1e4
"""

df = db.raw_sql(query)

bad_keywords = [
    "FUND", "ETF", "TRUST", "PORTFOLIO", "SERIES",
    "INVESTMENT", "SICAV"
]

df = df[~df['conm'].str.upper().str.contains('|'.join(bad_keywords))]

print(df.head())
print(f"Total unique CIKs: {len(df)}")

          cik                          conm  permno      market_cap  \
0  0000001750                      AAR CORP   54594       2215387.2   
1  0000001800           ABBOTT LABORATORIES   20482    191088014.13   
2  0000002230   ADAMS DIVERSIFIED EQUITY FD   10065       2139545.1   
3  0000002488        ADVANCED MICRO DEVICES   61241     238214560.0   
4  0000002969  AIR PRODUCTS & CHEMICALS INC   28222  60846024.17772   

         prc   dollar_volume  
0       62.4      15618033.6  
1     110.07    390781961.28  
2      17.71      1768857.09  
3     147.41   9130728411.58  
4  273.79999  242597743.1396  
Total unique CIKs: 1461


In [16]:
data = pd.read_parquet("bow_2010_batch_0.parquet")

data.head(100)

,accession,word,noccur
0,0000003453-10-000006,A&B,171.0
1,0000003453-10-000006,A&B-HAWAII,11.0
2,0000003453-10-000006,A&B/GENTRY,1.0
3,0000003453-10-000006,A&BS,153.0
4,0000003453-10-000006,A-1,2.0
...,...,...,...
95,0000003453-10-000006,ADEQUATE,8.0
96,0000003453-10-000006,ADJUST,3.0
97,0000003453-10-000006,ADJUSTED,12.0
98,0000003453-10-000006,ADJUSTMENT,11.0
